# 列联表独立性检验（Titanic）

> 对应代码：`MathStatsCode/code_in_notes/Chap.14/Titanic.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用泰坦尼克号乘客数据检验生存与性别是否独立（χ²(2) 检验）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.14`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.14")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all` 清空内存，载入泰坦尼克号乘客数据（含性别、登船人数、死亡人数）。本 notebook 检验**生存与性别是否独立**。


In [ ]:
%%stata
clear all
use ../datasets/titanic.dta


**按照性别加总**

`collapse (sum) num_aboard (sum) num_death, by(sex)`：把个体数据**聚合**到性别层面——`collapse` 用括号中的统计量（这里都是求和）产生新数据，每一行是一个性别。注意与 `egen ... by()` 的区别：collapse 会**替换**当前数据，只保留聚合结果。


In [ ]:
%%stata
collapse (sum)num_aboard (sum)num_death, by(sex)


**计算幸存人数**

`gen num_survive=num_aboard-num_death`：幸存人数=登船人数-死亡人数。至此得到2×2列联表的全部四个频数。


In [ ]:
%%stata
gen num_survive=num_aboard-num_death


**计算死亡总人数、幸存总人数**

计算边际合计：`qui: sum num_death` 得到死亡总人数存于宏 `death`，同理 `survive`，`N=death+survive` 为总人数。这些边际数在下一段用来计算**独立性假设下的理论频数**。


In [ ]:
%%stata
qui: sum num_death
local death=r(sum)
qui: sum num_survive
local survive=r(sum)
local N=`death'+`survive'


**计算理论死亡、幸存人数**

`gen e_num_death=num_aboard×death/N` 与 `gen e_num_survive=num_aboard×survive/N`：计算理论（期望）频数。理论公式为“行合计×列合计/总数”——在两变量独立的原假设下，每个格子中的期望人数应等于该性别的登船比例乘以死亡（生存）的总比例。


In [ ]:
%%stata
gen e_num_death=num_aboard*`death'/`N'
gen e_num_survive=num_aboard*`survive'/`N'


**计算检验统计量**

计算 **χ² 独立性检验统计量**：`gen chi2_death=(num_death-e_num_death)²/e_num_death`（死亡格的贡献）、`gen chi2_survive=(num_survive-e_num_survive)²/e_num_survive`（生存格），两者相加得到检验统计量；`su chi2` 求和后，用 `1-chi2(2, test_stat)` 计算 p 值。本数据把乘客分为儿童、男性、女性**三组**，与“死亡/生存”构成 3×2 列联表，自由度为 (3-1)×(2-1)=2。p 值小于显著性水平则拒绝“生存与性别独立”的原假设，认为生存状况与性别（年龄组）相关。


In [ ]:
%%stata
gen chi2_death=(num_death-e_num_death)^2/e_num_death
gen chi2_survive=(num_survive-e_num_survive)^2/e_num_survive
gen chi2=chi2_death+chi2_survive
qui: su chi2
local test_stat=r(sum)
di "test statistics= `test_stat'"
local p=1-chi2(2 ,`test_stat')
di "p-value= `p'" 
